# 04 — Reproducing Moutarde & Ultsch (2005)

Reproduces the U\*F clustering experiments from:

> Moutarde & Ultsch (2005). *U\*F clustering: a new performant cluster-mining method
> based on segmentation of Self-Organizing Maps.* WSOM 2005, Paris.

### Grid sizes (from §3.4 summary table)

The paper's §3.4 table lists grid sizes explicitly (rows × cols convention):

| Dataset | n | dims | Grid (rows × cols) | Topology |
|---|---|---|---|---|
| Atom | 800 | 3D | **50 × 82** | **toroidal** (IntraSOM `mapshape="toroid"`) |
| Lsun | 404 | 2D | **50 × 82** | planar |
| WingNut | 1016 | 2D | **50 × 82** | planar |
| ChainLink | 1000 | 3D | **50 × 82** | planar |
| TwoDiamonds | 800 | 2D | **40 × 50** | planar |

**Method:** z-score → ESOM → P-matrix (PDE, `pareto_fraction=0.2013`) →
`UStarFloodClustering.fit(u, p)` (U\* computed internally) → confusion matrix.
`threshold_anchor="upper"` throughout.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()


def load_fcps_fixture(name: str):
    for d in [Path.cwd(), *Path.cwd().parents]:
        p = d / 'tests' / 'fixtures' / 'fcps.npz'
        if p.is_file():
            z = np.load(p)
            return z[f'{name}_data'].copy(), z[f'{name}_cls'].copy()
    raise FileNotFoundError('tests/fixtures/fcps.npz not found')


from pyesom import ESOM, UStarFloodClustering
from pyesom.topology.pmatrix import compute_pmatrix
from pyesom.visualization.cluster import plot_ustar_cluster_overlay


def zscore(x):
    return (x - x.mean(axis=0)) / (x.std(axis=0) + 1e-9)


def confusion_matrix_df(true_labels, pred_labels):
    true_ids = sorted(np.unique(true_labels))
    pred_ids = sorted(np.unique(pred_labels))
    rows = []
    for t in true_ids:
        row = {'true \\ pred': t}
        for p in pred_ids:
            col = 'none' if p == -1 else str(p)
            row[col] = int(np.sum((true_labels == t) & (pred_labels == p)))
        rows.append(row)
    return pd.DataFrame(rows).set_index('true \\ pred')


print('Ready.')


Ready.


## 1  Atom — toroidal grid (IntraSOM)

Paper result: **perfect** (0 wrong, 0 unassigned).
Toroidal topology prevents border effects that would otherwise split one cluster.

MiniSom and TorchSOM only support rectangular/hexagonal; **IntraSOM** supports `mapshape='toroid'`.
Install: `pip install '.[intrasom]'`


In [2]:
data_atom, labels_atom = load_fcps_fixture('atom')
data_atom_z = zscore(data_atom)

som_atom = ESOM(50, 82, data_atom_z.shape[1], random_seed=42, backend='intrasom')
                # defaults: mapshape='toroid', lattice='hexa'
som_atom.fit(data_atom_z, epochs=20)

u_atom = som_atom.u_matrix()
p_atom = compute_pmatrix(som_atom.weights, data_atom_z, pareto_fraction=0.2013)

clf_atom = UStarFloodClustering(min_cluster_size=1, n_threshold_steps=100, threshold_anchor='lower')
clf_atom.fit(u_atom, p_atom)
pred_atom = clf_atom.predict(som_atom.bmu_indices(data_atom_z))

print(f'Clusters found : {clf_atom.n_clusters_}   threshold : {clf_atom.threshold_:.4f}')
print(f'Unassigned     : {(pred_atom == -1).sum()} / {len(pred_atom)}')
print('\nConfusion matrix  (paper: 400|0 / 0|400):')
print(confusion_matrix_df(labels_atom, pred_atom))


Using fontManager instance from /Users/mattijnvanhoek/.matplotlib/fontlist-v390.json


Loading dataframe...
Normalizing data...
Creating neighborhood...
Initializing map...


Creating Neuron Distance Rows:   0%|          | 0/50 [00:00<?, ?rows/s]

Starting Training...
Rough Training:


  0%|          | 0/10 [00:00<?, ?it/s]

Fine Tuning:


  0%|          | 0/10 [00:00<?, ?it/s]

Training completed successfully.
Clusters found : 3   threshold : 0.3737
Unassigned     : 6 / 800

Confusion matrix  (paper: 400|0 / 0|400):
             none    0    1    2
true \ pred                     
0               5    0  169  226
1               1  399    0    0


In [3]:
plot_ustar_cluster_overlay(
    clf_atom.ustar_, clf_atom.labels_,
    bmu_indices=som_atom.bmu_indices(data_atom_z),
    sample_labels=pred_atom,
    title=f'Atom — U*F overlay  (K={clf_atom.n_clusters_})',
    cell_pixels=5.0,
)


alt.LayerChart(...)

## 2  Lsun — planar grid

Paper result: 3 clusters, only 5 examples from cluster 3 left unassigned (1.2%).


In [4]:
data_lsun, labels_lsun = load_fcps_fixture('lsun')
data_lsun_z = zscore(data_lsun)

som_lsun = ESOM(50, 82, data_lsun_z.shape[1], random_seed=42, backend='intrasom')
som_lsun.fit(data_lsun_z, epochs=20)

u_lsun = som_lsun.u_matrix()
p_lsun = compute_pmatrix(som_lsun.weights, data_lsun_z, pareto_fraction=0.2013)

clf_lsun = UStarFloodClustering(min_cluster_size=1, n_threshold_steps=100, threshold_anchor='upper')
clf_lsun.fit(u_lsun, p_lsun)
pred_lsun = clf_lsun.predict(som_lsun.bmu_indices(data_lsun_z))

print(f'Clusters found : {clf_lsun.n_clusters_}   threshold : {clf_lsun.threshold_:.4f}')
print(f'Unassigned     : {(pred_lsun == -1).sum()} / {len(pred_lsun)}')
print('\nConfusion matrix  (paper: 200|0|0|0 / 0|100|0|0 / 0|0|95|5):')
print(confusion_matrix_df(labels_lsun, pred_lsun))


Loading dataframe...
Normalizing data...
Creating neighborhood...
Initializing map...


Creating Neuron Distance Rows:   0%|          | 0/50 [00:00<?, ?rows/s]

Starting Training...
Rough Training:


  0%|          | 0/10 [00:00<?, ?it/s]

Fine Tuning:


  0%|          | 0/10 [00:00<?, ?it/s]

Training completed successfully.
Clusters found : 5   threshold : 0.2525
Unassigned     : 2 / 404

Confusion matrix  (paper: 200|0|0|0 / 0|100|0|0 / 0|0|95|5):
             none    0   1  2
true \ pred                  
0               0  200   0  0
1               2   65  33  0
2               0  100   0  0
3               0    0   0  4


In [9]:
plot_ustar_cluster_overlay(
    clf_lsun.ustar_, clf_lsun.labels_,
    bmu_indices=som_lsun.bmu_indices(data_lsun_z),
    sample_labels=pred_lsun,
    title=f'Lsun — U*F overlay  (K={clf_lsun.n_clusters_})',
    cell_pixels=7.0, dot_size=40, dot_opacity=1
)


alt.LayerChart(...)

## 3  WingNut — planar grid

Paper result: 2 clusters, no misclassifications, ~9% unassigned.


In [13]:
data_wn, labels_wn = load_fcps_fixture('wingnut')
data_wn_z = zscore(data_wn)

som_wn = ESOM(50, 82, data_wn_z.shape[1], random_seed=42, backend='intrasom')
som_wn.fit(data_wn_z, epochs=20)

u_wn = som_wn.u_matrix()
p_wn = compute_pmatrix(som_wn.weights, data_wn_z, pareto_fraction=0.2013)

clf_wn = UStarFloodClustering(min_cluster_size=1, n_threshold_steps=100, threshold_anchor='upper')
clf_wn.fit(u_wn, p_wn)
pred_wn = clf_wn.predict(som_wn.bmu_indices(data_wn_z))

print(f'Clusters found : {clf_wn.n_clusters_}   threshold : {clf_wn.threshold_:.4f}')
print(f'Unassigned     : {(pred_wn == -1).sum()} / {len(pred_wn)}  ({100*(pred_wn==-1).mean():.1f}%)')
print('\nConfusion matrix  (paper: 455|0|45 / 0|456|44):')
print(confusion_matrix_df(labels_wn, pred_wn))


Loading dataframe...
Normalizing data...
Creating neighborhood...
Initializing map...


Creating Neuron Distance Rows:   0%|          | 0/50 [00:00<?, ?rows/s]

Starting Training...
Rough Training:


  0%|          | 0/10 [00:00<?, ?it/s]

Fine Tuning:


  0%|          | 0/10 [00:00<?, ?it/s]

Training completed successfully.
Clusters found : 8   threshold : 0.2727
Unassigned     : 104 / 1016  (10.2%)

Confusion matrix  (paper: 455|0|45 / 0|456|44):
             none    0   1  2  3  5  6
true \ pred                           
0              53  437   0  6  9  2  1
1              51  434  23  0  0  0  0


In [14]:
plot_ustar_cluster_overlay(
    clf_wn.ustar_, clf_wn.labels_,
    bmu_indices=som_wn.bmu_indices(data_wn_z),
    sample_labels=pred_wn,
    title=f'WingNut — U*F overlay  (K={clf_wn.n_clusters_})',
    cell_pixels=5.0,
)


alt.LayerChart(...)

## 4  ChainLink — planar grid

**§3.2** U\*F on U\*-matrix: paper gets 3 regions (one true cluster split), ~5% unassigned.

**§3.3 variant** U\*F on **U-matrix**: paper gets perfect result (2 clusters, 0 wrong, 0 unassigned).
Use `use_ustar=False` so `clf.fit(u, p)` treats U directly as the landscape.


In [15]:
data_cl, labels_cl = load_fcps_fixture('chainlink')
data_cl_z = zscore(data_cl)

som_cl = ESOM(50, 82, data_cl_z.shape[1], random_seed=42, backend='intrasom')
som_cl.fit(data_cl_z, epochs=20)

u_cl = som_cl.u_matrix()
p_cl = compute_pmatrix(som_cl.weights, data_cl_z, pareto_fraction=0.2013)
bmus_cl = som_cl.bmu_indices(data_cl_z)

# §3.2 — standard U*F on U*-matrix
clf_cl_ustar = UStarFloodClustering(min_cluster_size=1, n_threshold_steps=100,
                                     use_ustar=True, threshold_anchor='upper')
clf_cl_ustar.fit(u_cl, p_cl)
pred_cl_ustar = clf_cl_ustar.predict(bmus_cl)

# §3.3 variant — U*F on U-matrix
clf_cl_u = UStarFloodClustering(min_cluster_size=1, n_threshold_steps=100,
                                  use_ustar=False, threshold_anchor='upper')
clf_cl_u.fit(u_cl, p_cl)
pred_cl_u = clf_cl_u.predict(bmus_cl)

print('--- §3.2  U*F on U*-matrix (paper: 3 regions, one cluster split, ~5% unassigned) ---')
print(f'Clusters: {clf_cl_ustar.n_clusters_}   unassigned: {(pred_cl_ustar==-1).sum()}  ({100*(pred_cl_ustar==-1).mean():.1f}%)')
print(confusion_matrix_df(labels_cl, pred_cl_ustar))

print('\n--- §3.3  U*F on U-matrix (paper: 2 clusters, 0 wrong, 0 unassigned) ---')
print(f'Clusters: {clf_cl_u.n_clusters_}   unassigned: {(pred_cl_u==-1).sum()}')
print(confusion_matrix_df(labels_cl, pred_cl_u))


Loading dataframe...
Normalizing data...
Creating neighborhood...
Initializing map...


Creating Neuron Distance Rows:   0%|          | 0/50 [00:00<?, ?rows/s]

Starting Training...
Rough Training:


  0%|          | 0/10 [00:00<?, ?it/s]

Fine Tuning:


  0%|          | 0/10 [00:00<?, ?it/s]

Training completed successfully.
--- §3.2  U*F on U*-matrix (paper: 3 regions, one cluster split, ~5% unassigned) ---
Clusters: 6   unassigned: 7  (0.7%)
             none    0  1   2  3
true \ pred                     
0               5  448  0  46  1
1               2  491  7   0  0

--- §3.3  U*F on U-matrix (paper: 2 clusters, 0 wrong, 0 unassigned) ---
Clusters: 6   unassigned: 7
             none    0  1   2  3
true \ pred                     
0               5  448  0  46  1
1               2  491  7   0  0


In [16]:
alt.hconcat(
    plot_ustar_cluster_overlay(
        clf_cl_ustar.ustar_, clf_cl_ustar.labels_,
        bmu_indices=bmus_cl, sample_labels=pred_cl_ustar,
        title=f'§3.2 U*F on U*  (K={clf_cl_ustar.n_clusters_})',
        cell_pixels=5.0,
    ),
    plot_ustar_cluster_overlay(
        clf_cl_u.ustar_, clf_cl_u.labels_,
        bmu_indices=bmus_cl, sample_labels=pred_cl_u,
        title=f'§3.3 U*F on U   (K={clf_cl_u.n_clusters_})',
        cell_pixels=5.0,
    ),
).resolve_scale(color='independent')


alt.HConcatChart(...)

## 5  TwoDiamonds — smaller planar grid (40 × 50)

Paper uses a different, smaller grid `(40×50)` (explicitly listed in §3.4 table).
Paper result: 2 clusters, no misclassifications, ~12% unassigned.


In [17]:
data_td, labels_td = load_fcps_fixture('twodiamonds')
data_td_z = zscore(data_td)

som_td = ESOM(40, 50, data_td_z.shape[1], random_seed=42, backend='intrasom')
som_td.fit(data_td_z, epochs=20)

u_td = som_td.u_matrix()
p_td = compute_pmatrix(som_td.weights, data_td_z, pareto_fraction=0.2013)

clf_td = UStarFloodClustering(min_cluster_size=1, n_threshold_steps=100, threshold_anchor='upper')
clf_td.fit(u_td, p_td)
pred_td = clf_td.predict(som_td.bmu_indices(data_td_z))

print(f'Clusters found : {clf_td.n_clusters_}   threshold : {clf_td.threshold_:.4f}')
print(f'Unassigned     : {(pred_td == -1).sum()} / {len(pred_td)}  ({100*(pred_td==-1).mean():.1f}%)')
print('\nConfusion matrix  (paper: 259|0|41 / 0|270|30):')
print(confusion_matrix_df(labels_td, pred_td))


Loading dataframe...
Normalizing data...
Creating neighborhood...
Initializing map...


Creating Neuron Distance Rows:   0%|          | 0/40 [00:00<?, ?rows/s]

Starting Training...
Rough Training:


  0%|          | 0/10 [00:00<?, ?it/s]

Fine Tuning:


  0%|          | 0/10 [00:00<?, ?it/s]

Training completed successfully.
Clusters found : 3   threshold : 0.2323
Unassigned     : 77 / 800  (9.6%)

Confusion matrix  (paper: 259|0|41 / 0|270|30):
             none    0  1   2
true \ pred                  
0              28  342  0  30
1              49  349  2   0


In [18]:
plot_ustar_cluster_overlay(
    clf_td.ustar_, clf_td.labels_,
    bmu_indices=som_td.bmu_indices(data_td_z),
    sample_labels=pred_td,
    title=f'TwoDiamonds — U*F overlay  (K={clf_td.n_clusters_})',
    cell_pixels=7.0,
)


alt.LayerChart(...)

## Summary

Fill in the 'Our' columns after running:

| Dataset | Paper K | Paper unassigned | Paper wrong | Our K | Our unassigned | Our wrong |
|---|---|---|---|---|---|---|
| Atom (toroidal) | 2 | 0% | 0% | | | |
| Lsun | 3 | 1.2% | 0% | | | |
| WingNut | 2 | 9% | 0% | | | |
| ChainLink (U\*) | 3\* | 5.5% | 0% | | | |
| ChainLink (U) | 2 | 0% | 0% | | | |
| TwoDiamonds | 2 | 12% | 0% | | | |

\* ChainLink with U\* splits one true cluster in two — the extra region is entirely within a real group.

**Potential sources of deviation:**
- MiniSom sequential stochastic training vs paper's (unspecified) SOM implementation
- PDE uses `pareto_fraction=0.2013`; paper uses Ultsch's own PDE implementation
- `threshold_anchor='upper'` used here; paper's exact threshold-edge choice is ambiguous
